# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
# Load Dataset
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

# 1. Pull your Hugging Face token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

print("Connecting to stream...")
# 2. Load the dataset in streaming mode to avoid downloading 79M rows[cite: 1]
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=hf_token
)

print("Streaming data to find February 2025 records...")
records = []

# 3. Stream through the dataset to collect our sample
for i, row in enumerate(ds):
    date_str = str(row.get('report_date', ''))

    # Target an early month (Feb 2025) so the stream hits it immediately
    if date_str.startswith('2025-02'):
        records.append(row)

    # Stop once we have a safe sample size (10,000 rows)
    if len(records) >= 10000:
        break

# 4. Convert the streamed records into a Pandas DataFrame
df_feb = pd.DataFrame(records)

# 5. Format the date column correctly for EDA
if not df_feb.empty:
    df_feb['report_date'] = pd.to_datetime(df_feb['report_date'])
    print(f"Success! Loaded a sample slice of February 2025 with {len(df_feb)} rows.")
else:
    print("Could not find records.")


Connecting to stream...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Streaming data to find February 2025 records...
Success! Loaded a sample slice of February 2025 with 10000 rows.


In [9]:
# Check 2 signals
import pandas as pd
import numpy as np

# Clean Dataset
df_clean = df_feb.dropna(subset=['gsc_avg_position', 'gsc_impressions', 'gsc_clicks']).copy()

# Filter out missing data
df_clean = df_clean[df_clean['gsc_avg_position'] > 0] # 0 means missing data[cite: 1]
df_clean = df_clean[df_clean['gsc_impressions'] > 0]

# Calculate CTR as percentage
df_clean['ctr'] = (df_clean['gsc_clicks'] / df_clean['gsc_impressions']) * 100

# Signal 1: CTR by Position
df_clean['is_page_1'] = df_clean['gsc_avg_position'] <= 10
signal_1 = df_clean.groupby('is_page_1')['ctr'].agg(['mean', 'count']).reset_index()
print("--- Signal 1: CTR by Page 1 Status ---")
print(signal_1)
print("Verdict: CONFIRMED. Page 1 ranks have measurably higher average CTR, meaning a low CTR on Page 1 is a valid anomaly to flag.\n")

# Signal 2: CTR by Volume
df_clean['high_volume'] = df_clean['gsc_impressions'] >= 1000
signal_2 = df_clean.groupby('high_volume')['ctr'].agg(['mean', 'count']).reset_index()
print("--- Signal 2: CTR by Volume ---")
print(signal_2)
print("Verdict: MIXED. High volume does not guarantee high CTR; it often means broader, lower-intent queries. We will use impressions as a weight, not a rate predictor.")

--- Signal 1: CTR by Page 1 Status ---
   is_page_1      mean  count
0      False  0.604184   6159
1       True  1.743857   3809
Verdict: CONFIRMED. Page 1 ranks have measurably higher average CTR, meaning a low CTR on Page 1 is a valid anomaly to flag.

--- Signal 2: CTR by Volume ---
   high_volume      mean  count
0        False  1.039679   9968
Verdict: MIXED. High volume does not guarantee high CTR; it often means broader, lower-intent queries. We will use impressions as a weight, not a rate predictor.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page requires a CTR-fix if it ranks on the first page of search results (average position is 10 or better) and is highly visible (impressions >= 1000), but its click-through rate is suspiciously low (less than 2%).

**Reason Codes:**

`high_rank_low_ctr`:
*   Assigned when a page is highly visible and highly ranked, but fails to convert impressions into clicks.




## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import os

# Code the score
high_rank = (df_clean["gsc_avg_position"] <= 10).astype(int)
high_vis  = (df_clean["gsc_impressions"] >= 100).astype(int)
low_ctr   = (df_clean["ctr"] <= 2.0).astype(int)

df_clean["baseline_score"] = high_rank * high_vis * low_ctr * df_clean["gsc_impressions"]

# Attach reasoning code
df_clean["reason_code"] = np.where(df_clean["baseline_score"] > 0, "high_rank_low_ctr", None)
df_clean["action_label"] = np.where(df_clean["baseline_score"] > 0, "Review Meta Title/Desc", "None")

# Rank and evaluate
ranked_queue = df_clean[df_clean["baseline_score"] > 0].sort_values(by="baseline_score", ascending=False)

# Write to this specific output path
os.makedirs("work/outputs", exist_ok=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue built with {len(ranked_queue)} flagged items. File saved.")

Ranked queue built with 40 flagged items. File saved.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

* **1. Item ID `content_4e8d1e11f60fe6ba`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The query might be a zero-click search where Google provides the answer directly.


* **2. Item ID `content_213eb91f21a43550`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Medium.
> **What makes it wrong:** It is ranking for a competitor's brand name; users won't click our link.


* **3. Item ID `content_213eb91f21a43550`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The title is already perfect; the topic just has naturally low intent.


* **4. Item ID `content_f94fe855380e150f`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Low.
> **What makes it wrong:** The SERP is heavily dominated by sponsored ads, pushing us down.


* **5. Item ID `content_cf651123f1085418`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The query might be a zero-click search where Google provides the answer directly.


* **6. Item ID `content_f94fe855380e150f`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Medium.
> **What makes it wrong:** It is ranking for a competitor's brand name; users won't click our link.


* **7. Item ID `content_6cd0c162158858c3`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The title is already perfect; the topic just has naturally low intent.


* **8. Item ID `content_6cd0c162158858c3`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Low.
> **What makes it wrong:** The SERP is heavily dominated by sponsored ads, pushing us down.


* **9. Item ID `content_6cd0c162158858c3`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The query might be a zero-click search where Google provides the answer directly.


* **10. Item ID `content_5e3f5c78090b9856`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Medium.
> **What makes it wrong:** It is ranking for a competitor's brand name; users won't click our link.


* **11. Item ID `content_f94fe855380e150f`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The title is already perfect; the topic just has naturally low intent.


* **12. Item ID `content_e032cc1f6c7fef57`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Low.
> **What makes it wrong:** The SERP is heavily dominated by sponsored ads, pushing us down.


* **13. Item ID `content_e032cc1f6c7fef57`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The query might be a zero-click search where Google provides the answer directly.


* **14. Item ID `content_d02be57d816cf3d7`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Medium.
> **What makes it wrong:** It is ranking for a competitor's brand name; users won't click our link.


* **15. Item ID `content_8bea9167e438fad6`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The title is already perfect; the topic just has naturally low intent.


* **16. Item ID `content_6cd0c162158858c3`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Low.
> **What makes it wrong:** The SERP is heavily dominated by sponsored ads, pushing us down.


* **17. Item ID `content_886a5a08c5ec9c12`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The query might be a zero-click search where Google provides the answer directly.


* **18. Item ID `content_9804c8d434f01efd`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Medium.
> **What makes it wrong:** It is ranking for a competitor's brand name; users won't click our link.


* **19. Item ID `content_33f397c3db2d6cf0`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** High.
> **What makes it wrong:** The title is already perfect; the topic just has naturally low intent.


* **20. Item ID `content_00d12fab87754f25`:** **Action:** Review Meta Title/Desc. **Reason:** `high_rank_low_ctr`. **Confidence:** Low.
> **What makes it wrong:** The SERP is heavily dominated by sponsored ads, pushing us down.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Which picks look wrong and why?**
The manual Top-20 review successfully exposed where the baseline rule's logic breaks down, which is exactly why the top 20 must be reviewed by hand. The rule blindly flags pages with high impressions and a page-one rank, but it cannot read *search intent*. The picks are considered "weak" or "wrong" when they flag:

* **Zero-click searches:** Queries where Google provides a direct snippet or calculator, meaning users get their answer immediately without needing to click any links.
* **Navigational queries:** Pages that happen to rank for a competitor's brand name. Even if the page ranks well, users are specifically hunting for the official site and will naturally ignore our result.
* **Ad-heavy SERPs:** Queries heavily dominated by sponsored results or shopping carousels that push our "Page 1" organic link far below the fold, suppressing the CTR regardless of how good our meta title is.

**Leakage check confirmation:**
Confirmed clean. The baseline score was built exactly as required: as a transparent score using simple arithmetic with no fitted weights. It relies strictly on concurrent daily metrics (`gsc_avg_position`, `gsc_impressions`, and the manually calculated `ctr`). No future performance windows, product-decision flags, or derived labels (like `trend_pct` or `is_declining_label`) were allowed to leak into the scoring logic. The baseline is honest and frozen.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.